# 🏥 Diabetic Retinopathy Detection — Starting Kit

Welcome to the **Diabetic Retinopathy Detection Challenge**!

This notebook will guide you through:
1. **Exploratory Data Analysis (EDA)** — understanding the dataset
2. **Evaluation Metrics** — how your submissions are scored
3. **Baseline Model** — a simple starting point
4. **Creating a Submission** — how to format and submit your solution

---
## 0. Setup

In [ ]:
import os
import csv
import random
import numpy as np
from PIL import Image
import matplotlib.pyplot as plt
from collections import Counter

# Paths — adjust if needed
TRAIN_DIR = "dataset/train"
TEST_DIR = "dataset/test"
CLASSES_FILE = "_classes.csv"

SEVERITY_NAMES = {
    "0": "No DR",
    "1": "Mild",
    "2": "Moderate",
    "3": "Severe",
    "4": "Proliferative DR"
}

---
## 1. Exploratory Data Analysis (EDA)

### 1.1 Dataset Overview

The dataset comes from the **IDRiD (Indian Diabetic Retinopathy Image Dataset)**, sourced via [Roboflow](https://universe.roboflow.com/officeworkspace/diabetic-retinopathy-dataset) under a **CC BY 4.0** license.

It contains **510 retinal fundus photographs** annotated with one of **5 severity levels** of diabetic retinopathy (DR):

| Label | Severity | Description |
|-------|----------|-------------|
| 0 | No DR | Healthy retina |
| 1 | Mild | Microaneurysms only |
| 2 | Moderate | More than microaneurysms, less than severe |
| 3 | Severe | Extensive hemorrhages, venous beading |
| 4 | Proliferative DR | Neovascularization, vitreous hemorrhage |

In [ ]:
def load_classes(split_dir):
    """Load the _classes.csv file and return rows as list of dicts."""
    path = os.path.join(split_dir, CLASSES_FILE)
    with open(path) as f:
        reader = csv.DictReader(f, skipinitialspace=True)
        return list(reader)

train_data = load_classes(TRAIN_DIR)
test_data = load_classes(TEST_DIR)

print(f"Training samples: {len(train_data)}")
print(f"Test samples:     {len(test_data)}")
print(f"Total:            {len(train_data) + len(test_data)}")
print(f"\nColumns: {list(train_data[0].keys())}")
print(f"\nExample row: {train_data[0]}")

### 1.2 Class Distribution

Let's check how the severity levels are distributed across the training set.

In [ ]:
label_cols = ["0", "1", "2", "3", "4"]

def get_class_counts(data):
    counts = {}
    for c in label_cols:
        counts[c] = sum(1 for r in data if r[c].strip() == "1")
    return counts

train_counts = get_class_counts(train_data)
test_counts = get_class_counts(test_data)

# Print distribution
print("Class Distribution (Training Set):")
print("-" * 50)
for c in label_cols:
    pct = train_counts[c] / len(train_data) * 100
    bar = "█" * int(pct)
    print(f"  Class {c} ({SEVERITY_NAMES[c]:>18s}): {train_counts[c]:>4d}  ({pct:5.1f}%)  {bar}")

print(f"\n  Total: {sum(train_counts.values())} labels for {len(train_data)} samples")

In [ ]:
# Bar chart
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

colors = ["#4CAF50", "#FFC107", "#FF9800", "#F44336", "#9C27B0"]
severity_labels = [f"{c} — {SEVERITY_NAMES[c]}" for c in label_cols]

# Train
bars1 = axes[0].bar(severity_labels, [train_counts[c] for c in label_cols], color=colors, edgecolor="white")
axes[0].set_title("Training Set Distribution", fontsize=14, fontweight="bold")
axes[0].set_ylabel("Number of images")
axes[0].set_ylim(0, max(train_counts.values()) * 1.2)
for bar, c in zip(bars1, label_cols):
    axes[0].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 3,
                f"{train_counts[c]}", ha="center", fontweight="bold")

# Test
bars2 = axes[1].bar(severity_labels, [test_counts[c] for c in label_cols], color=colors, edgecolor="white")
axes[1].set_title("Test Set Distribution", fontsize=14, fontweight="bold")
axes[1].set_ylabel("Number of images")
axes[1].set_ylim(0, max(test_counts.values()) * 1.2)
for bar, c in zip(bars2, label_cols):
    axes[1].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 1,
                f"{test_counts[c]}", ha="center", fontweight="bold")

for ax in axes:
    ax.tick_params(axis="x", rotation=30)

plt.tight_layout()
plt.show()

**Key observations:**
- The dataset is **imbalanced**: classes 0 (No DR) and 2 (Moderate) dominate (~33% each).
- **Class 1 (Mild) is very rare** (~5%), making it the hardest to predict.
- Classes 3 (Severe) and 4 (Proliferative) are also under-represented.
- Each image has **exactly one label** (single-label classification), encoded as a one-hot vector.

⚠️ The class imbalance is a key challenge — consider using **class weights**, **oversampling**, or **data augmentation** to address it.

### 1.3 Sample Images

Let's visualize some retinal images from each severity class.

In [ ]:
def get_label(row):
    """Return the active label index for a row."""
    for c in label_cols:
        if row[c].strip() == "1":
            return c
    return None

# Group train images by class
class_images = {c: [] for c in label_cols}
for row in train_data:
    label = get_label(row)
    if label:
        class_images[label].append(row["filename"])

# Show 2 random samples per class
fig, axes = plt.subplots(2, 5, figsize=(20, 8))
fig.suptitle("Sample Retinal Images by Severity Level", fontsize=16, fontweight="bold")

for col, c in enumerate(label_cols):
    samples = random.sample(class_images[c], min(2, len(class_images[c])))
    for row_idx, fname in enumerate(samples):
        img_path = os.path.join(TRAIN_DIR, fname)
        if os.path.exists(img_path):
            img = Image.open(img_path)
            axes[row_idx, col].imshow(img)
        axes[row_idx, col].axis("off")
        if row_idx == 0:
            axes[row_idx, col].set_title(f"Class {c}\n{SEVERITY_NAMES[c]}",
                                         fontsize=12, fontweight="bold", color=colors[col])

plt.tight_layout()
plt.show()

### 1.4 Image Properties

In [ ]:
# Check a few image sizes
widths, heights = [], []
sample_files = [row["filename"] for row in train_data[:50]]

for fname in sample_files:
    img_path = os.path.join(TRAIN_DIR, fname)
    if os.path.exists(img_path):
        img = Image.open(img_path)
        widths.append(img.size[0])
        heights.append(img.size[1])

print(f"Sample of {len(widths)} images:")
print(f"  Width  — min: {min(widths)}, max: {max(widths)}, mean: {np.mean(widths):.0f}")
print(f"  Height — min: {min(heights)}, max: {max(heights)}, mean: {np.mean(heights):.0f}")

# File sizes
sizes = []
for fname in [row["filename"] for row in train_data]:
    fpath = os.path.join(TRAIN_DIR, fname)
    if os.path.exists(fpath):
        sizes.append(os.path.getsize(fpath))

sizes_mb = [s / 1024 / 1024 for s in sizes]
print(f"\nFile sizes:")
print(f"  Min: {min(sizes_mb):.2f} MB")
print(f"  Max: {max(sizes_mb):.2f} MB")
print(f"  Mean: {np.mean(sizes_mb):.2f} MB")
print(f"  Total dataset: {sum(sizes_mb):.1f} MB")

---
## 2. Evaluation Metrics

Your submissions are evaluated using **three metrics** plus the runtime:

### 2.1 F1 Score (Macro)

The **F1 Macro** score computes the F1 score for each class independently, then takes the **unweighted average**. This treats all classes equally, regardless of their frequency.

$$F1_{\text{macro}} = \frac{1}{C} \sum_{i=1}^{C} F1_i = \frac{1}{C} \sum_{i=1}^{C} \frac{2 \cdot \text{Precision}_i \cdot \text{Recall}_i}{\text{Precision}_i + \text{Recall}_i}$$

This is the **primary metric** for the leaderboard. Because of the class imbalance (class 1 is rare), this metric penalizes models that ignore minority classes.

### 2.2 F1 Score (Micro)

The **F1 Micro** score computes a global F1 by counting total true positives, false positives, and false negatives across all classes.

$$F1_{\text{micro}} = \frac{2 \cdot TP_{\text{global}}}{2 \cdot TP_{\text{global}} + FP_{\text{global}} + FN_{\text{global}}}$$

This metric favors majority classes and is equivalent to overall accuracy in the single-label case.

### 2.3 Hamming Loss

The **Hamming loss** measures the fraction of labels that are incorrectly predicted:

$$\text{Hamming Loss} = \frac{1}{N \cdot L} \sum_{i=1}^{N} \sum_{j=1}^{L} \mathbb{1}[\hat{y}_{ij} \neq y_{ij}]$$

where $N$ is the number of samples and $L = 5$ is the number of labels. Lower is better.

### 2.4 Thresholding

Your model outputs **probabilities** (values between 0 and 1) for each of the 5 classes. These are **thresholded at 0.5** to produce binary predictions before computing the metrics.

### 2.5 Runtime

The total time for training + inference is also reported.

In [ ]:
# Demonstration of the metrics
from sklearn.metrics import f1_score, hamming_loss

# Example: simulate some predictions
np.random.seed(42)
n_samples = 20
n_classes = 5

# True labels (one-hot)
y_true = np.zeros((n_samples, n_classes), dtype=int)
for i in range(n_samples):
    y_true[i, np.random.randint(0, n_classes)] = 1

# Simulated predictions (noisy)
y_pred_proba = y_true * 0.7 + (1 - y_true) * 0.1 + np.random.randn(n_samples, n_classes) * 0.15
y_pred_proba = np.clip(y_pred_proba, 0, 1)
y_pred_bin = (y_pred_proba >= 0.5).astype(int)

f1_mac = f1_score(y_true, y_pred_bin, average="macro", zero_division=0)
f1_mic = f1_score(y_true, y_pred_bin, average="micro", zero_division=0)
hl = hamming_loss(y_true, y_pred_bin)

print("=== Example Metric Computation ===")
print(f"  F1 Macro:     {f1_mac:.4f}")
print(f"  F1 Micro:     {f1_mic:.4f}")
print(f"  Hamming Loss: {hl:.4f}")

---
## 3. Baseline Model

The baseline uses a **ResNet-18** pretrained on ImageNet, fine-tuned on the training data.

Key details:
- Input size: 128×128 pixels
- Loss: `BCEWithLogitsLoss` (multi-label binary cross-entropy)
- Optimizer: Adam, lr = 1e-4
- Epochs: 10
- Data augmentation: random flips + rotations

The baseline code is in [`solution/submission.py`](solution/submission.py). Here's the key interface:

In [ ]:
# Show the submission interface
print("""
=== Required Submission Interface ===

Your submission.py must define:

    def get_model():
        return Model()

    class Model:
        def fit(self, X_train, y_train, train_img_dir):
            \"\"\"Train the model.
            
            Args:
                X_train: pd.DataFrame with column 'filename'
                y_train: pd.DataFrame with columns ['0','1','2','3','4'] (one-hot labels)
                train_img_dir: str, path to the directory containing training images
            \"\"\"
            ...

        def predict(self, X_test, test_img_dir):
            \"\"\"Generate predictions.
            
            Args:
                X_test: pd.DataFrame with column 'filename'
                test_img_dir: str, path to the directory containing test images
            
            Returns:
                np.ndarray of shape (n_samples, 5) with probabilities in [0, 1]
            \"\"\"
            ...
""")

---
## 4. Creating a Submission

To submit your solution:

1. Create a `submission.py` file following the interface above.
2. **Zip** it:
   ```bash
   zip submission.zip submission.py
   ```
3. Upload `submission.zip` on the CodaBench competition page.

### Tips for improvement:
- Use a **larger input resolution** (e.g., 224×224 or 512×512)
- Try **other architectures** (EfficientNet, DenseNet, ViT)
- Apply **class weights** in the loss function to handle imbalance
- Use **more aggressive data augmentation** (color jitter, elastic transforms)
- Try **learning rate scheduling** (cosine annealing, warm restarts)
- Implement **cross-validation** for more robust training

---

Good luck! 🍀